# Kata 27 — Multi-Pass Review e Independent Reviewer

> Spec: [`specs/027-multi-pass-review/spec.md`](../../specs/027-multi-pass-review/spec.md)  |  Arquitectura: [`ARCHITECTURE.md`](../../ARCHITECTURE.md)

## Setup

Si `ANTHROPIC_API_KEY` no está en el entorno, se pedirá aquí (sin echo).

In [ ]:
from katas._shared.bootstrap import bootstrap
import json

client, settings = bootstrap(model="claude-sonnet-4-6", budget_calls=10)
print("modelo:", settings.model)

## 1. Concepto en mis palabras

El modelo que generó código retiene contexto de su razonamiento — eso lo
hace mal revisor de su propio output. Una instancia independiente
detecta más issues. Para PRs grandes: per-file pass + cross-file
integration pass.

## 2. Por qué importa

Self-review produce reviews superficiales o auto-justificantes. Single-
pass sobre 14 archivos dispersa atención: feedback inconsistente, bugs
omitidos.

## 3. Independent reviewer (sesión nueva)

In [ ]:
# Paso 1: el modelo genera código — esa sesión va a tener su razonamiento.
gen_resp = client.messages.create(
    system="Eres un dev. Implementa lo pedido.",
    messages=[{"role":"user","content":"""Implementa una función Python `process_payment(amount, user_id)` que:
- Valida amount > 0.
- Loguea la transacción.
- Devuelve un dict con status y txn_id.
Hazla compacta."""}],
)
generated_code = next((b.text for b in gen_resp.content if b.type == "text"), "")
print("=== código generado ===")
print(generated_code[:600])

In [ ]:
# Paso 2 — Self-review (mala práctica): MISMA sesión revisa su output.
self_review = client.messages.create(
    system="Eres un dev. Implementa lo pedido.",
    messages=[
        {"role":"user","content":"Implementa process_payment como antes."},
        {"role":"assistant","content": generated_code},
        {"role":"user","content":"Ahora revisa críticamente el código que produjiste. Lista issues."},
    ],
)
print("=== self-review (mala práctica) ===")
print(next((b.text for b in self_review.content if b.type == "text"), "")[:600])

In [ ]:
# Paso 3 — Independent review (buena práctica): SESIÓN NUEVA, sin razonamiento previo.
indep_review = client.messages.create(
    system="Eres un reviewer estricto. Lista issues con file:line:severity:rationale. No defiendas el código.",
    messages=[{"role":"user","content": f"""Revisa este código Python:

```python
{generated_code}
```

Lista issues de seguridad, robustez y manejo de errores."""}],
)
print("=== independent review (sesión nueva) ===")
print(next((b.text for b in indep_review.content if b.type == "text"), "")[:1000])

Comparando los dos reviews: el self-review tiende a justificar decisiones
del autor; el independent review las cuestiona. Mismo modelo, diferente
contexto.

## 4. Multi-pass para PR grande — per-file + cross-file

In [ ]:
# Simulamos un PR de 3 archivos con dependencia cross-file.
FILES = {
    "src/auth.py": """def get_user(user_id: int) -> dict:
    return db.find_user(user_id)  # puede ser None""",
    "src/billing.py": """from src.auth import get_user

def charge_user(user_id: int, amount: float):
    user = get_user(user_id)
    db.charge(user["card_id"], amount)  # null deref si user es None""",
    "src/notify.py": """def send_receipt(user_id: int, amount: float):
    user = get_user(user_id)
    email_service.send(user.email, amount)""",
}

# Pass A: per-file, cada uno en su sesión nueva
def review_file(path, content):
    r = client.messages.create(
        system="Lista local issues sólo. NO especules sobre callers.",
        messages=[{"role":"user","content": f"```python\n# {path}\n{content}\n```"}],
    )
    return {"path": path, "review": next((b.text for b in r.content if b.type=='text'), "")}

per_file = [review_file(p, c) for p, c in FILES.items()]
print("=== Pass A: per-file ===")
for r in per_file:
    print(f"\n[{r['path']}]")
    print(r["review"][:200])

In [ ]:
# Pass B: cross-file, ve sólo los outputs del pass A (NO el código crudo)
summary = json.dumps([{"path": r["path"], "issues_summary": r["review"][:500]} for r in per_file], ensure_ascii=False)

cross_file = client.messages.create(
    system="Recibirás summaries por archivo. Tu trabajo es detectar interacciones cross-file (data flow, null prop, race conditions). Lista issues nuevos que cada review individual no podía ver.",
    messages=[{"role":"user","content": summary}],
)
print("=== Pass B: cross-file ===")
print(next((b.text for b in cross_file.content if b.type == "text"), "")[:800])

El cross-file pass tiene que detectar el null deref de `user["card_id"]`
y `user.email` que se origina en `get_user` retornando None. Eso es
imposible de ver mirando un solo archivo aislado.

## 5. Argumento de certificación

- **Self-review** ⊂ misma sesión = revisor sesgado por razonamiento.
- **Independent review** = sesión nueva sin generación previa = mejor
  detección.
- **Multi-pass para PRs grandes**: per-file (profundidad local) +
  cross-file (interacciones).
- **Quórum 2/3 entre reviews independientes** parece sólido pero
  suprime issues raros legítimos. Antipatrón disfrazado.
- Conexión con Kata 12 (chaining): el cross-file pass es exactamente
  el integration pass del prompt chaining.

## 6. Auto-evaluación

**1. ¿Por qué el quórum de N reviews independientes hace daño?**

Issues genuinos que aparecen sólo a veces (por non-determinism del
modelo) son los **menos confiables de detectar a la primera** y los
**más valiosos** cuando aparecen. Filtrar por consenso los descarta.

**2. ¿Cómo aseguro que el reviewer no tiene la cadena de razonamiento
del generador?**

Sesión nueva. `messages=[]` empieza desde cero, sólo el código resultante
en `user`. No incluir el system prompt original ni el assistant turn
de generación.

**3. ¿Cuándo el cross-file pass detecta algo que el per-file no?**

Null prop entre módulos, race conditions en chains de funciones,
contracts implícitos rotos (caller asume que callee garantiza X). Todo
lo que requiere mirar dos archivos a la vez.